
## LAB | Using MCP in LangChain
## Step 1: Setup and Installation

Objective: Install required packages and set up your environment.



In [1]:
pip install langchain langchain-openai langchain-mcp-adapters mcp python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
pip show langchain-mcp-adapters

Name: langchain-mcp-adapters
Version: 0.2.2
Summary: Make Anthropic Model Context Protocol (MCP) tools compatible with LangChain and LangGraph agents.
Home-page: 
Author: 
Author-email: Vadym Barda <19161700+vbarda@users.noreply.github.com>
License-Expression: MIT
Location: c:\Users\User\anaconda3\Lib\site-packages
Requires: langchain-core, mcp, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from openai import OpenAI  # ✅ Added missing import

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")  # ✅ Extract the key
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Create a .env file with your key.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
client = OpenAI(api_key=api_key)  # ✅ Now this works

print("✅ Environment and OpenAI client configured.")
print("🛡️ No API calls made. Safe to run repeatedly.")

✅ Environment and OpenAI client configured.
🛡️ No API calls made. Safe to run repeatedly.


## Step 2: Connect to MCP Server
Objective: Establish connection to an MCP server. This can be any server of your choice. We recommend starting small and connecting to a local filesystem MCP server first.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio

# Configure the LangChain documentation MCP server
# MultiServerMCPClient takes a dict of server configs
# Each config specifies transport type and connection details
mcp_client = MultiServerMCPClient({
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
    }
})

print("MCP client configured for LangChain documentation server")
print(f"Server: langchain-docs → https://docs.langchain.com/mcp")

MCP client configured for LangChain documentation server
Server: langchain-docs → https://docs.langchain.com/mcp


## Step 3: Load MCP Tools into LangChain
Objective: Convert MCP tools to LangChain tools for use in agents.

In [5]:
# Load tools from the MCP server
# The client is stateless: get_tools() creates ephemeral sessions under the hood
mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} tools from MCP server(s)")
print("\nAvailable tools:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

Loaded 2 tools from MCP server(s)

Available tools:
  - search_docs_by_lang_chain: Search across the Docs by LangChain knowledge base to find relevant information,...
  - query_docs_filesystem_docs_by_lang_chain: Run a read-only shell-like query against a virtualized, in-memory filesystem roo...


## Step 4: Create Agent with MCP Tools
Objective: Build a LangChain agent that uses MCP tools.

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Get tools from MCP server
mcp_tools = await mcp_client.get_tools()

# Create a prompt that explains the MCP capabilities the agent can use
system_prompt = """
You are a helpful assistant with access to MCP tools for LangChain documentation.
Available capabilities:
- search_docs_by_lang_chain: search the docs knowledge base for relevant answers.
- query_docs_filesystem_docs_by_lang_chain: inspect documentation pages directly.
Use the MCP tools whenever the question depends on current docs or exact LangChain guidance.
""".strip()

# Create the agent with the model, prompt, and MCP tools
agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt=system_prompt,
)

print(f"Agent created with {len(mcp_tools)} MCP tools")
print(f"Tools: {[tool.name for tool in mcp_tools]}")

Agent created with 2 MCP tools
Tools: ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']


In [7]:
# Query the agent about LangChain
question = "How do I create a LangChain agent with tools?"
print(f"Question: {question}\n")

result = await agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print(f"Answer: {result['messages'][-1].content}")

Question: How do I create a LangChain agent with tools?

Answer: To create a LangChain agent with tools, you can use the `create_agent` function, which is the standard way to build agents in LangChain. This function allows you to specify a language model and a list of tools that the agent can use. Here's a basic outline of how to do this:

### Step 1: Import Necessary Modules
You need to import the `create_agent` function and any tools you want to use.

```python
from langchain.agents import create_agent
```

### Step 2: Define Your Tools
You can define your tools as plain Python functions or coroutines. You can also use the `@tool` decorator to customize tool properties.

```python
from langchain.tools import tool

@tool
def my_tool(input):
    # Your tool logic here
    return "Result from my_tool"
```

### Step 3: Create the Agent
You can create the agent by calling `create_agent`, passing in the model and the tools.

#### Example with Static Model
```python
agent = create_agent(
  

In [8]:
# Smoke test: ask a question that should require MCP documentation lookup.
test_query = (
    "Use the LangChain docs MCP tools to find the current guidance for create_agent "
    "and summarize how to pass a prompt and tools. Mention which MCP tool(s) you used."
)

result = await agent.ainvoke({"messages": [HumanMessage(content=test_query)]})

print("=== Message Trace ===")
for message in result["messages"]:
    role = getattr(message, "type", message.__class__.__name__)
    content = getattr(message, "content", "")
    tool_calls = getattr(message, "tool_calls", None)
    print(f"{role}: {str(content)[:300]}")
    if tool_calls:
        print(f"  tool_calls: {[call.get('name') for call in tool_calls]}")

print("\n=== Final Answer ===")
print(result["messages"][-1].content)


=== Message Trace ===
human: Use the LangChain docs MCP tools to find the current guidance for create_agent and summarize how to pass a prompt and tools. Mention which MCP tool(s) you used.
ai: 
  tool_calls: ['search_docs_by_lang_chain']
tool: [{'type': 'text', 'text': 'Title: Create an evaluator\nLink: https://docs.langchain.com/langsmith/evaluators#create-an-evaluator\nPage: langsmith/evaluators\nContent: Create an evaluator\nIn the LangSmith UI , select Evaluators in the left sidebar. Click + Evaluator to open the Add Evaluator panel. 
ai: 
  tool_calls: ['query_docs_filesystem_docs_by_lang_chain']
tool: [{'type': 'text', 'text': 'exit: 0\n--- stdout ---\nWhat\'s new in LangChain v1\nLangChain v1 is a focused, production-ready foundation for building agents. We\'ve streamlined the framework around three core improvements: create_agent The new standard for building agents in LangChain, replacing lang
ai: To create an agent using LangChain, you can utilize the `create_agent` function

## Step 5: Access MCP Resources
Objective: Use MCP resources to provide context to your agent.

In [13]:
# Access MCP resources
# MultiServerMCPClient exposes get_resources(), which returns LangChain Blob objects.
# Alias it here to match the exercise wording.
get_langchain_resources = mcp_client.get_resources

mcp_resources = await get_langchain_resources()

print(f"Loaded {len(mcp_resources)} MCP resources")
for resource in mcp_resources[:5]:
    print(f"  - {resource.metadata.get('uri', 'unknown')}")


Loaded 1 MCP resources
  - mintlify://skills/langchain


In [14]:
def read_resource(resource):
    """Read a LangChain Blob returned by the MCP adapter."""
    return resource.as_string()


resource_snippets = []
for resource in mcp_resources[:3]:
    uri = resource.metadata.get("uri", "unknown")
    content = read_resource(resource).strip()
    resource_snippets.append(f"URI: {uri}\n{content[:1200]}")

resource_context = "\n\n---\n\n".join(resource_snippets) if resource_snippets else "No MCP resources were returned."

print("Resource context preview:\n")
print(resource_context[:2000])


Resource context preview:

URI: mintlify://skills/langchain
---
name: Langchain
description: Use when building AI agents and applications with LLMs, integrating tools and models, creating multi-agent systems, or deploying agents to production. Reach for LangChain when you need to quickly build agents with tool calling, memory, streaming, and observability.
metadata:
    mintlify-proj: langchain
    version: "1.0"
---

# LangChain Skill

## Product summary

LangChain is an open-source framework for building agents and LLM applications. It provides a prebuilt agent architecture, integrations for hundreds of LLMs (OpenAI, Anthropic, Google, etc.), and tools for orchestrating complex workflows. Key files and commands:

- **Core packages**: `langchain` (framework), `langchain-openai`, `langchain-anthropic`, etc. (provider integrations)
- **Key imports**: `from langchain.agents import create_agent`, `from langchain.tools import tool`, `from langchain.chat_models import init_chat_model`
- **C

In [15]:
# Use MCP resources as background context for the agent.
resource_system_prompt = f"""
You are a helpful assistant with read-only MCP resource context.
Use the resource context below as background information when it is sufficient.
If the answer is not in the provided context, say so clearly.

MCP resource context:
{resource_context}
""".strip()

resource_agent = create_agent(
    model=llm,
    system_prompt=resource_system_prompt,
)

resource_question = "Summarize the loaded MCP resources and explain how they can help answer LangChain questions."
print(f"Question: {resource_question}\n")

resource_result = await resource_agent.ainvoke({
    "messages": [HumanMessage(content=resource_question)]
})

print("Answer:")
print(resource_result["messages"][-1].content)


Question: Summarize the loaded MCP resources and explain how they can help answer LangChain questions.

Answer:
The loaded MCP resource provides information about LangChain, an open-source framework designed for building AI agents and applications using large language models (LLMs). It includes a prebuilt agent architecture, integrations with various LLM providers (such as OpenAI and Anthropic), and tools for managing complex workflows.

Key features of LangChain include:

- **Core Packages**: The framework itself (`langchain`) and specific integrations for different LLMs (e.g., `langchain-openai`).
- **Key Imports**: Essential components for building applications, such as creating agents and initializing chat models.
- **CLI Tool**: The `langsmith` CLI for deployment and observability of agents.
- **Documentation**: A primary resource for understanding how to use LangChain effectively.

This information can help answer questions about how to use LangChain for building AI applications,

## Step 6: Build a Complete MCP-Enabled Agent
Objective: Create a functional agent that demonstrates MCP capabilities.

In [18]:
from langchain.agents import create_agent

# Build a complete MCP-enabled agent using the existing LangChain docs MCP client.
mcp_tools = await mcp_client.get_tools()

complete_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="""You are a helpful assistant that answers questions about LangChain by using MCP tools when needed.
Prefer the docs tools for current documentation, exact API usage, and implementation details.""".strip(),
)

complete_query = "Create 5 quiz questions from the documents."
print(f"Question: {complete_query}\n")

complete_result = await complete_agent.ainvoke({
    "messages": [HumanMessage(content=complete_query)]
})

print("=== Message Trace ===")
for message in complete_result["messages"]:
    role = getattr(message, "type", message.__class__.__name__)
    content = getattr(message, "content", "")
    tool_calls = getattr(message, "tool_calls", None)
    print(f"{role}: {str(content)[:300]}")
    if tool_calls:
        print(f"  tool_calls: {[call.get('name') for call in tool_calls]}")

print("\nFinal Answer:")
print(complete_result["messages"][-1].content)


Question: Create 5 quiz questions from the documents.

=== Message Trace ===
human: Create 5 quiz questions from the documents.
ai: 
  tool_calls: ['search_docs_by_lang_chain']
tool: [{'type': 'text', 'text': 'Title: Test 1: Handle off-topic questions\nLink: https://docs.langchain.com/langsmith/test-react-agent-pytest#test-1-handle-off-topic-questions\nPage: langsmith/test-react-agent-pytest\nContent: Test 1: Handle off-topic <mark><b>questions</b></mark>\nThe first test will be
ai: 
  tool_calls: ['query_docs_filesystem_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']
tool: [{'type': 'text', 'text': 'exit: 0\n--- stdout ---\nTest a ReAct agent with Pytest/Vitest and LangSmith\nThis tutorial will show you how to use LangSmith\'s integrations with popular testing tools (Pytest, Vitest, and Jest) to evaluate your LLM application. We will crea

In [19]:
# Follow-up question using the same complete MCP-enabled agent.
follow_up_query = "Which MCP tool should I use when I need the full contents of a documentation page instead of a search hit?"
print(f"Question: {follow_up_query}\n")

follow_up_result = await complete_agent.ainvoke({
    "messages": [HumanMessage(content=follow_up_query)]
})

print("Answer:")
print(follow_up_result["messages"][-1].content)


Question: Which MCP tool should I use when I need the full contents of a documentation page instead of a search hit?

Answer:
You should use the `query_docs_filesystem` tool to get the full contents of a documentation page. You can do this by passing the specific page path with a `.mdx` extension to the `cat` command. For example, you would use `cat /path/to/documentation.mdx` to read the entire content of that page.


## Step 7 — MCP vs Direct API Integration

MCP provides a standardized protocol for connecting AI systems to external tools and resources, making integrations reusable and easier to maintain across frameworks like LangChain. Direct API integration gives more control and can provide better performance, but requires custom code for every service and increases maintenance complexity. MCP is best for scalable multi-tool ecosystems, while direct APIs are useful for single specialized integrations.

| Feature                  | MCP Integration    | Direct API Integration |
| ------------------------ | ------------------ | ---------------------- |
| Setup Complexity         | Medium             | Simple                 |
| Scalability              | High               | Low                    |
| Standardization          | Excellent          | Poor                   |
| Reusability              | High               | Low                    |
| Maintenance              | Easier             | Harder                 |
| Performance              | Slight overhead    | Faster                 |
| Multi-system Integration | Excellent          | Difficult              |
| Flexibility              | High               | Medium                 |
| Best For                 | Enterprise systems | Small projects         |


## Step 8: OPTIONAL, Build a Practical Example

# MCP + LangChain Lab

## Overview

This project demonstrates integrating MCP servers with LangChain agents.

## Features

- Connect to MCP filesystem server
- Load MCP tools into LangChain
- Build AI agent using MCP tools
- Access MCP resources
- Document analysis example

## Installation

```bash
pip install langchain
pip install langchain-openai
pip install langchain-mcp-adapters
pip install mcp
pip install python-dotenv